In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = pd.read_csv('diabetes.csv')

In [ ]:
df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [ ]:
df.corr()['Outcome']

,Outcome
Pregnancies,0.221898
Glucose,0.466581
BloodPressure,0.065068
SkinThickness,0.074752
Insulin,0.130548
BMI,0.292695
DiabetesPedigreeFunction,0.173844
Age,0.238356
Outcome,1.000000


Before training any nural network we need to scaling all of them.

In [ ]:
x = df.iloc[:,:-1].values
y = df.iloc[:,-1].values

In [ ]:
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
X = sc.fit_transform(x)

In [ ]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test = train_test_split(X,y,random_state=42)

Now this is tim eto make a model (Nueral Network)

In [ ]:
import tensorflow
from tensorflow import keras
from keras import Sequential
from keras.layers import Dense, Dropout

In [ ]:
model = Sequential()
model.add(Dense(24,activation='relu',input_dim=x_train.shape[1]))
model.add(Dense(1,activation='sigmoid'))

model.compile(optimizer = 'adam',loss='binary_crossentropy',metrics=['accuracy'])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:
model.fit(x_train, y_train,batch_size=32, epochs=25, validation_data=(x_test,y_test))

Epoch 1/25
18/18 ━━━━━━━━━━━━━━━━━━━━ 2s 15ms/step - accuracy: 0.5226 - loss: 0.6941 - val_accuracy: 0.6354 - val_loss: 0.6489
Epoch 2/25
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.6424 - loss: 0.6400 - val_accuracy: 0.6719 - val_loss: 0.6146
Epoch 3/25
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6979 - loss: 0.5995 - val_accuracy: 0.7083 - val_loss: 0.5911
Epoch 4/25
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7326 - loss: 0.5708 - val_accuracy: 0.7188 - val_loss: 0.5737
Epoch 5/25
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7344 - loss: 0.5481 - val_accuracy: 0.7135 - val_loss: 0.5609
Epoch 6/25
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7552 - loss: 0.5313 - val_accuracy: 0.6979 - val_loss: 0.5510
Epoch 7/25
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7517 - loss: 0.5171 - val_accuracy: 0.6979 - val_loss: 0.5431
Epoch 8/25
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7587 - loss: 0.5059 - val_accuracy: 0.7083 - val_loss

How to automate the model to find out the appropiate parametres for out model...

1. How to selelct appropiate Optimizer
2. Number of nodes in a layer
3. How to select number of layes
4. All in one

In [ ]:
# If you don't have keras tuner first install it.
pip install -U keras-tuner

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.4/129.4 kB 2.8 MB/s eta 0:00:00


In [ ]:
import keras_tuner as kt

In [ ]:
def build_model(hp):
  model = Sequential()
  model.add(Dense(24,activation='relu',input_dim=x_train.shape[1]))
  model.add(Dense(1,activation='sigmoid'))

  optimizer = hp.Choice('optimizer',['adam','rmsprop','sgd','adagrad'])
  model.compile(optimizer = optimizer,loss='binary_crossentropy',metrics=['accuracy'])
  return model

In [ ]:
tuner = kt.RandomSearch(build_model,
                        objective='val_accuracy',
                        max_trials=5)

Reloading Tuner from ./untitled_project/tuner0.json


In [ ]:
tuner.search(x_train,y_train,epochs=5,validation_data =(x_test,y_test),verbose = 1)

In [ ]:
tuner.get_best_hyperparameters(num_trials = 3)[0].values

{'optimizer': 'rmsprop'}

In [ ]:
model = tuner.get_best_models(num_models=1)[0]

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 2 variables whereas the saved optimizer has 6 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [ ]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 24)             │           216 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            25 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 241 (964.00 B)

 Trainable params: 241 (964.00 B)

 Non-trainable params: 0 (0.00 B)

In [ ]:
model.fit(x_train,y_train,batch_size=32,epochs=100,initial_epoch = 6, validation_data=(x_test,y_test))
#initial_epoch means we initially train the model till epochs=5 i need to retain the information

Epoch 7/100
18/18 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.7587 - loss: 0.5466 - val_accuracy: 0.7448 - val_loss: 0.5584
Epoch 8/100
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7743 - loss: 0.5228 - val_accuracy: 0.7500 - val_loss: 0.5451
Epoch 9/100
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7812 - loss: 0.5067 - val_accuracy: 0.7448 - val_loss: 0.5347
Epoch 10/100
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7847 - loss: 0.4936 - val_accuracy: 0.7396 - val_loss: 0.5271
Epoch 11/100
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7830 - loss: 0.4831 - val_accuracy: 0.7448 - val_loss: 0.5221
Epoch 12/100
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7882 - loss: 0.4752 - val_accuracy: 0.7396 - val_loss: 0.5189
Epoch 13/100
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7917 - loss: 0.4684 - val_accuracy: 0.7396 - val_loss: 0.5162
Epoch 14/100
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7934 - loss: 0.4625 - val_accuracy: 0.73

In [ ]:
#trying to find best number of neurons in one hidden layer
def build_model(hp):
  model  = Sequential()
  units = hp.Int('units',min_value = 8,max_value = 128,step=8) # number of neurons will be taken = 8,16,24,32,40......
  model.add(Dense(units=units,activation='relu',input_dim=x_train.shape[1]))
  model.add(Dense(1,activation='sigmoid'))

  model.compile(optimizer = 'rmsprop',loss = 'binary_crossentropy',metrics=['accuracy'])
  return model

In [ ]:
tuner = kt.RandomSearch(
    build_model,
    objective='val_accuracy',
    max_trials=5,
    directory='units_tuning',
    project_name='units_tuning_project' # Use a new project name
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:
tuner.search(x_train,y_train,epochs=5,validation_data=(x_test,y_test),verbose=1)

Trial 5 Complete [00h 00m 02s]
val_accuracy: 0.7135416865348816

Best val_accuracy So Far: 0.765625
Total elapsed time: 00h 00m 13s


In [ ]:
tuner.get_best_hyperparameters()[0].values

{'units': 104}

In [ ]:
#trying to find best number of hidden layers in our model
def build_model(hp):
  model = Sequential()
  model.add(Dense(104,activation='relu',input_dim=8))
  for i in range(hp.Int('num_layer', min_value=1,max_value=10)):
    model.add(Dense(104,activation='relu'))
  model.add(Dense(1,activation='sigmoid'))

  model.compile(optimizer = 'rmsprop',loss = 'binary_crossentropy',metrics=['accuracy'])
  return model

In [ ]:
tuner = kt.RandomSearch(build_model,
                        objective ="val_accuracy",
                        max_trials=2,
                        directory='layers',
                        project_name='number_of_layers')

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:
tuner.search(x_train,y_train,epochs=5,validation_data=(x_test,y_test),verbose=1)

Trial 2 Complete [00h 00m 04s]
val_accuracy: 0.765625

Best val_accuracy So Far: 0.765625
Total elapsed time: 00h 00m 10s


In [ ]:
tuner.get_best_hyperparameters()[0].values

{'num_layer': 4}

Now we are doing more than one thing

In [ ]:
def build_model(hp):
  model = Sequential()

  counter = 0
  for i in range(hp.Int('num_layer', min_value=1,max_value=10)):
    if counter == 0:
      model.add(
          Dense(
              hp.Int('units'+str(i),min_value = 8,max_value = 128,step=8),
              activation=hp.choice('activation' + str(i)), values=['relu','sigmoid','tanh']),
              input_dims=8)
    else:
       model.add(
          Dense(
              hp.Int('units'+str(i),min_value = 8,max_value = 128,step=8),
              activation=hp.choice('activation' + str(i)), values=['relu','sigmoid','tanh']))
    counter += 1

  model.add(Dense(1,activation='sigmoid'))

  model.compile(optimizer = 'rmsprop',loss = 'binary_crossentropy',metrics=['accuracy'])
  return model

In [ ]:
tuner= kt.RandomSearch(
    build_model,
    objective='val_accuracy',
    max_trials=5,
    directory = "all",
    project_name = "neurons_&_nodes"
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:
tuner.search(x_train,y_train,epochs=5,validation_data=(x_test,y_test),verbose=1)

Trial 5 Complete [00h 00m 07s]
val_accuracy: 0.703125

Best val_accuracy So Far: 0.765625
Total elapsed time: 00h 00m 21s


In [ ]:
tuner.get_best_hyperparameters()[0].values

{'num_layer': 3,
 'units0': 120,
 'activation0': 'tanh',
 'units1': 8,
 'activation1': 'relu',
 'units2': 8,
 'activation2': 'relu'}

In [ ]:
final_model = tuner.get_best_models(num_models=1)[0]

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 2 variables whereas the saved optimizer has 10 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [ ]:
history = final_model.fit(x_train,y_train,epochs=100,initial_epoch = 6, validation_data=(x_test,y_test))

Epoch 7/100
18/18 ━━━━━━━━━━━━━━━━━━━━ 2s 18ms/step - accuracy: 0.7795 - loss: 0.4428 - val_accuracy: 0.7552 - val_loss: 0.5291
Epoch 8/100
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7934 - loss: 0.4351 - val_accuracy: 0.7552 - val_loss: 0.5323
Epoch 9/100
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7847 - loss: 0.4324 - val_accuracy: 0.7500 - val_loss: 0.5341
Epoch 10/100
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7951 - loss: 0.4295 - val_accuracy: 0.7500 - val_loss: 0.5382
Epoch 11/100
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7899 - loss: 0.4276 - val_accuracy: 0.7500 - val_loss: 0.5335
Epoch 12/100
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.8021 - loss: 0.4264 - val_accuracy: 0.7448 - val_loss: 0.5379
Epoch 13/100
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7934 - loss: 0.4245 - val_accuracy: 0.7448 - val_loss: 0.5358
Epoch 14/100
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.8003 - loss: 0.4232 - val_accuracy: 0.75

In [ ]:
def build_model(hp):
  model = Sequential()

  counter = 0
  for i in range(hp.Int('num_layer', min_value=1,max_value=10)):
    if counter == 0:
      model.add(
          Dense(
              hp.Int('units'+str(i),min_value = 8,max_value = 128,step=8),
              activation=hp.Choice('activation' + str(i), ['relu','sigmoid','tanh']),
              input_dim=8))
      model.add(Dropout(hp.Choice('dropout'+str(i), [0.2,0.4,0.5,0.6])))
    else:
       model.add(
          Dense(
              hp.Int('units'+str(i),min_value = 8,max_value = 128,step=8),
              activation=hp.Choice('activation' + str(i), ['relu','sigmoid','tanh'])))
       model.add(Dropout(hp.Choice('dropout'+str(i), [0.2,0.4,0.5,0.6])))
    counter += 1

  model.add(Dense(1,activation='sigmoid'))

  model.compile(optimizer = 'rmsprop',loss = 'binary_crossentropy',metrics=['accuracy'])
  return model

In [ ]:
tuner= kt.RandomSearch(
    build_model,
    objective='val_accuracy',
    max_trials=5,
    directory = "all_dropout",
    project_name = "neurons_&_nodes_dropout"
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:
tuner.search(x_train,y_train,epochs=5,validation_data=(x_test,y_test),verbose=1)

Trial 5 Complete [00h 00m 05s]
val_accuracy: 0.640625

Best val_accuracy So Far: 0.6979166865348816
Total elapsed time: 00h 00m 23s


In [ ]:
tuner.get_best_hyperparameters()[0].values

{'num_layer': 2,
 'units0': 8,
 'activation0': 'relu',
 'dropout0': 0.6,
 'units1': 128,
 'activation1': 'tanh',
 'dropout1': 0.5,
 'units2': 112,
 'activation2': 'sigmoid',
 'dropout2': 0.5,
 'units3': 88,
 'activation3': 'relu',
 'dropout3': 0.6,
 'units4': 96,
 'activation4': 'relu',
 'dropout4': 0.5,
 'units5': 88,
 'activation5': 'relu',
 'dropout5': 0.2}